In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

df = pd.read_csv("calorie_dataset.csv")
df.head()

,protein_g,carbs_g,fat_g,fiber_g,sugar_g,calories,calorie_class
0,22.5,65.2,13.7,9.9,26.7,479.0,medium
1,15.9,20.7,16.8,7.6,15.0,316.0,low
2,28.3,62.4,14.5,10.8,15.3,524.0,medium
3,23.3,18.6,7.9,17.0,0.2,250.0,low
4,30.8,55.1,31.9,12.5,7.2,602.0,medium


In [ ]:
features = ["protein_g", "carbs_g", "fat_g", "fiber_g", "sugar_g"]
X = df[features]
y = df["calories"]     

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (4000, 5) Test: (1000, 5)


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), features)]
)

lr_pipe = Pipeline([
    ("pre", preprocessor),
    ("model", LinearRegression())
])

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

lr_rmse = -cross_val_score(lr_pipe, X_train, y_train, cv=cv,
                           scoring="neg_root_mean_squared_error")
lr_mae  = -cross_val_score(lr_pipe, X_train, y_train, cv=cv,
                           scoring="neg_mean_absolute_error")
lr_r2   =  cross_val_score(lr_pipe, X_train, y_train, cv=cv, scoring="r2")

print("Linear Regression — CV metrics")
print("CV RMSE:", lr_rmse.mean(), "±", lr_rmse.std())
print("CV MAE :", lr_mae.mean(),  "±", lr_mae.std())
print("CV R²  :", lr_r2.mean(),   "±", lr_r2.std())

Linear Regression — CV metrics
CV RMSE: 30.039295448345193 ± 0.38418099231497976
CV MAE : 23.910297420167286 ± 0.41320706106586735
CV R²  : 0.9357296778089488 ± 0.0025545356715396413


In [ ]:
lr_pipe.fit(X_train, y_train)

Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['protein_g', 'carbs_g',
                                                   'fat_g', 'fiber_g',
                                                   'sugar_g'])])),
                ('model', LinearRegression())])

In [ ]:
y_pred = lr_pipe.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae  = mean_absolute_error(y_test, y_pred)
test_r2   = r2_score(y_test, y_pred)

print("Linear Regression — Test metrics")
print("RMSE:", test_rmse)
print("MAE :", test_mae)
print("R²  :", test_r2)

Linear Regression — Test metrics
RMSE: 29.667068194316265
MAE : 24.055351050157864
R²  : 0.93335392180662


In [ ]:
import pandas as pd
import os

df = pd.read_csv("calorie_dataset.csv")

df_processed = df[["protein_g", "carbs_g", "fat_g", "fiber_g", "sugar_g", "calories"]]

os.makedirs("data/processed", exist_ok=True)
df_processed.to_csv("data/processed/calorie_dataset_processed.csv", index=False)

print("Original shape :", df.shape)
print("Processed shape:", df_processed.shape)
print("Columns        :", df_processed.columns.tolist())
print("Missing values :", df_processed.isnull().sum().sum())
print("Duplicates     :", df_processed.duplicated().sum())

Original shape : (5000, 7)
Processed shape: (5000, 6)
Columns        : ['protein_g', 'carbs_g', 'fat_g', 'fiber_g', 'sugar_g', 'calories']
Missing values : 0
Duplicates     : 0
